# Clase 01 · Introducción al PLN y ciclo de una solución

22 de septiembre de 2026 · 15:00–17:00 · Versión 2026-09-23.

Práctica formativa, 0%.

## Objetivo y contrato

Distinguir clasificación, extracción y generación; definir entrada, salida y criterio de éxito. Aquí clasificamos mensajes ficticios de soporte en acceso, pagos, conectividad o software. La categoría orienta una revisión; no autoriza cambios de cuenta.

Datos: `solicitudes.csv`, corpus sintético original de 144 filas. `text` es el mensaje; `label` es su categoría de referencia; `family` agrupa variantes relacionadas. No lo usamos para afirmar rendimiento productivo.

## Entorno

**Colab:** abre https://colab.research.google.com, sube este notebook y los datos indicados al panel Archivos.


In [1]:
from pathlib import Path
import sys, platform, json, re, unicodedata
import pandas as pd
from importlib.metadata import version
print("Python:", platform.python_version(), "pandas:", version("pandas"))
# Local: busca 07_datos en el curso. Colab: sube los archivos al panel Archivos.
candidates = [Path.cwd(), *Path.cwd().parents]
DATA = next((p / '07_datos' for p in candidates if (p / '07_datos/solicitudes.csv').exists()), Path.cwd())
if not (DATA / 'solicitudes.csv').exists():
    raise FileNotFoundError('Sube solicitudes.csv al panel Archivos de Colab, o abre el notebook dentro de curso_pln_tec.')
OUT = Path.cwd() / 'resultados_clase01'
OUT.mkdir(exist_ok=True)


Python: 3.12.10 pandas: 3.0.5


## Paso 1 · Cargar y leer una tabla

`pd.read_csv` lee un archivo separado por comas. `df` será un DataFrame: tabla con una fila por mensaje y una columna por variable. `head(5)` permite revisar las primeras cinco filas; no resume todo el corpus.

In [2]:
df = pd.read_csv(DATA / 'solicitudes.csv', encoding='utf-8')
print(df[['id', 'text', 'label']].head(5).to_string(index=False))
print('Filas del corpus:', len(df))

  id                                                                                    text  label
S001                                          Olvidé la contraseña y necesito restablecerla. acceso
S002           Buen día, olvidé la contraseña y necesito restablecerla. ¿Me pueden orientar? acceso
S003 Olvidé la contraseña y necesito restablecerla; ya reinicié y sigue igual, urge tantito. acceso
S004                                         El código de verificación no llega a mi correo. acceso
S005          Buen día, el código de verificación no llega a mi correo. ¿Me pueden orientar? acceso
Filas del corpus: 144


## Paso 2 · Anotar antes de mirar la referencia

`sample(12, random_state=42)` conserva el mismo sorteo en este entorno. En una columna propia anota la categoría probable de cada mensaje. Define entrada, salida, una regla posible y su límite. Si hay dos problemas, indica qué información falta: la referencia es una decisión de anotación, no una verdad infalible.

In [3]:
sample = df.sample(12, random_state=42).copy()
print(sample[['id', 'text']].to_string(index=False))
anotaciones_personales = {}  # Completa id: etiqueta y justifica dudas en Markdown.

  id                                                                                      text
S118                                                Necesito activar la licencia del software.
S020                    Buen día, el portal rechaza mi nombre de usuario. ¿Me pueden orientar?
S083                   Buen día, no puedo conectarme a la VPN desde casa. ¿Me pueden orientar?
S098                    Buen día, la red de invitados no permite navegar. ¿Me pueden orientar?
S057        El importe no coincide con la cotización; ya reinicié y sigue igual, urge tantito.
S013                                                    No puedo entrar con el segundo factor.
S133                                    Quiero volver a una versión anterior de la aplicación.
S066 El banco confirmó el cargo pero no tengo recibo; ya reinicié y sigue igual, urge tantito.
S067                                                  La factura tiene un impuesto incorrecto.
S019                                              

## Paso 3 · Entender una regla pequeña

Una función recibe `texto` y devuelve una categoría. `lower()` permite buscar sin distinguir mayúsculas. `in` comprueba presencia literal; no interpreta intención. Si aparece «contraseña», la regla devuelve acceso; en otro caso pide revisión.

Predice la salida de «Ya cambié la contraseña; el pago sigue fallando». La explicación está después de ejecutar.

In [4]:
def regla_demo(texto):
    if 'contraseña' in texto.lower():
        return 'acceso'
    return 'revisar'
print(regla_demo('Ya cambié la contraseña; el pago sigue fallando.'))

acceso


### Explicación del caso guiado

Devuelve acceso por coincidencia literal. Sin embargo, el problema vigente del mensaje es el pago. Una palabra asociada con acceso no implica que esa sea la petición actual. La regla necesita alcance y evaluación con casos negativos.

## Paso 4 · Ejecutar la regla de cuatro categorías

Se mantiene el comportamiento de la regla original del curso. `any(...)` comprueba si al menos una pista está presente. Se prueban pagos, acceso y conectividad en ese orden; software es el valor por defecto. Esa prioridad es una decisión explícita y puede causar errores. No hay entrenamiento: las pistas las escribe una persona.

In [5]:
def normalize(texto):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', texto)).strip()

def rule_label(texto):
    t = normalize(texto).lower()
    reglas = [
        ('pagos', ['factura', 'pago', 'pagó', 'cobro', 'cargo', 'tarjeta', 'reembolso']),
        ('acceso', ['contrase', 'sesión', 'usuario', 'código', 'codigo', 'cuenta', 'acceso']),
        ('conectividad', ['wifi', 'red', 'internet', 'vpn', 'conexión'])
    ]
    for etiqueta, pistas in reglas:
        if any(pista in t for pista in pistas):
            return etiqueta
    return 'software'

sample['prediction'] = sample['text'].map(rule_label)
print(sample[['id', 'text', 'label', 'prediction']].to_string(index=False))

  id                                                                                      text        label   prediction
S118                                                Necesito activar la licencia del software.     software     software
S020                    Buen día, el portal rechaza mi nombre de usuario. ¿Me pueden orientar?       acceso       acceso
S083                   Buen día, no puedo conectarme a la VPN desde casa. ¿Me pueden orientar? conectividad conectividad
S098                    Buen día, la red de invitados no permite navegar. ¿Me pueden orientar? conectividad conectividad
S057        El importe no coincide con la cotización; ya reinicié y sigue igual, urge tantito.        pagos     software
S013                                                    No puedo entrar con el segundo factor.       acceso     software
S133                                    Quiero volver a una versión anterior de la aplicación.     software     software
S066 El banco confirmó el cargo 

## Paso 5 · Interpretar desacuerdos

Compara la salida `prediction` con `label` para cada fila. Un desacuerdo puede señalar un fallo de la regla o una etiqueta discutible: lee el texto. Cuenta aciertos para esta muestra y explica qué solicitud quedaría mal dirigida. Este conteo no demuestra generalización.

In [6]:
errors = sample[sample['label'] != sample['prediction']]
print('Aciertos en la muestra:', int((sample['label'] == sample['prediction']).sum()), '/', len(sample))
print(errors[['id','text','label','prediction']].to_string(index=False))
print('Caso de contraste:', rule_label('No se trata de mi tarjeta: no puedo entrar a la red.'))

Aciertos en la muestra: 10 / 12
  id                                                                               text  label prediction
S057 El importe no coincide con la cotización; ya reinicié y sigue igual, urge tantito.  pagos   software
S013                                             No puedo entrar con el segundo factor. acceso   software
Caso de contraste: pagos


### Explicación del contraste

El caso devuelve pagos porque «tarjeta» se examina primero, aunque se niega como problema y se menciona acceso a la red. La regla no modela negación ni relaciones. Conserva ese error al valorar una mejora: un total puede ocultarlo.

## Paso 6 · Reto original: un cambio y dos fallos

Escribe dos mensajes propios donde falle la regla. Propón un solo cambio, vuelve a probar los mismos 12 IDs y añade un contraejemplo que pueda empeorar. Conserva la comparación anterior. Identifica un falso positivo (asigna una clase indebidamente) y un falso negativo (no reconoce un caso de esa clase). No basta con aumentar aciertos: explica el límite.

In [9]:
# Mensaje 1 -> No contiene palabras del diccionario de pagos, pero forma parte de esta área. 
# Mensaje 2 -> 'Es un caso no bien definido por el usuario, por lo que no hay forma de categorizarlo por las reglas establecidas. 

mensaje_1 = "No reconoce el depósito realizado en OXXO, urge respuesta para activación"
mensaje_2 = "No puedo abrir el archivo adjunto que me enviaron"

# Modificar regla
def rule_label_v2(texto):
    t = normalize(texto).lower()
    reglas = [
        ('pagos', ['factura', 'pago', 'pagó', 'cobro', 'cargo', 'tarjeta',
                   'reembolso', 'depósito', 'deposito', 'depósito']),   # Depósito es el cambio propuesto
        ('acceso', ['contrase', 'sesión', 'usuario', 'código', 'codigo', 'cuenta', 'acceso']),
        ('conectividad', ['wifi', 'red', 'internet', 'vpn', 'conexión'])
    ]
    for etiqueta, pistas in reglas:
        if any(pista in t for pista in pistas):
            return etiqueta
    return 'software'

sample['prediction_v2'] = sample['text'].map(rule_label_v2)

comparacion = sample[['id', 'text', 'label', 'prediction', 'prediction_v2']].copy()
comparacion['cambio'] = comparacion.apply(
    lambda r: 'igual' if r['prediction'] == r['prediction_v2'] else 'CAMBIÓ', axis=1
)

aciertos_v1 = int((sample['label'] == sample['prediction']).sum())
aciertos_v2 = int((sample['label'] == sample['prediction_v2']).sum())

print("Comparación v1 vs v2 sobre los mismos 12 IDs")
print(comparacion.to_string(index=False))
print(f"\nAciertos v1: {aciertos_v1}/12")
print(f"Aciertos v2: {aciertos_v2}/12")

# Contraejemplo
contraejemplo = "El depósito de la fianza no se refleja, pero tampoco puedo entrar a la red wifi"
print("\nContraejemplo")
print("Texto      :", contraejemplo)
print("v1         :", rule_label(contraejemplo)) 
print("v2         :", rule_label_v2(contraejemplo))

# Falso positivo y falso negativo
falso_negativo = {
    'texto': mensaje_1,
    'label': 'pagos',
    'pred_v1': rule_label(mensaje_1),      
    'pred_v2': rule_label_v2(mensaje_1),
    'tipo': 'Falso negativo de pagos en v1'
}
falso_positivo = {
    'texto': contraejemplo,
    'label': 'conectividad',
    'pred_v1': rule_label(contraejemplo),      
    'pred_v2': rule_label_v2(contraejemplo),   
    'tipo': 'Falso positivo de pagos en v2'
}

print("\nFalso negativo (v1)")
for k, v in falso_negativo.items():
    print(f"  {k}: {v}")

print("\nFalso positivo (v2)")
for k, v in falso_positivo.items():
    print(f"  {k}: {v}")

argumento = (
    "El cambio NO basta: (a) no mejora los 12 IDs originales (sigue 10/12), "
    "solo arregla un caso inventado; (b) introduce un falso positivo cuando "
    "coexisten 'depósito' y 'wifi', porque la regla no modela negación ni "
    "intención principal; (c) el mismo total puede ocultar errores distintos. "
    "Añadir una palabra es una mejora local, no una solución: haría falta "
    "priorizar por intención, detectar negación o pasar a un modelo supervisado."
)
print("\nArgumento")
print(argumento)

Comparación v1 vs v2 sobre los mismos 12 IDs
  id                                                                                      text        label   prediction prediction_v2 cambio
S118                                                Necesito activar la licencia del software.     software     software      software  igual
S020                    Buen día, el portal rechaza mi nombre de usuario. ¿Me pueden orientar?       acceso       acceso        acceso  igual
S083                   Buen día, no puedo conectarme a la VPN desde casa. ¿Me pueden orientar? conectividad conectividad  conectividad  igual
S098                    Buen día, la red de invitados no permite navegar. ¿Me pueden orientar? conectividad conectividad  conectividad  igual
S057        El importe no coincide con la cotización; ya reinicié y sigue igual, urge tantito.        pagos     software      software  igual
S013                                                    No puedo entrar con el segundo factor.       ac

## Paso 7 · Guardar y cerrar

`to_csv` guarda la tabla; `index=False` evita añadir el índice de pandas como dato. Conserva además el notebook ejecutado con tus razones. En Colab descarga el CSV desde el panel Archivos.

Comprensión: ¿el programa aprendió de `label`? No: solo la usó para comparar; las reglas estaban escritas de antemano. Un modelo supervisado sí ajustaría parámetros a partir de ejemplos, lo que estudiaremos después.

In [10]:
sample.to_csv(OUT / 'regla_resultados.csv', index=False, encoding='utf-8')
print('Archivo:', (OUT / 'regla_resultados.csv').name)
assert len(sample) == 12 and set(sample['label']) <= {'acceso','pagos','conectividad','software'}

Archivo: regla_resultados.csv


## Continuidad y referencias

Ahora sabemos definir el problema y observar fallos. Clase 2 preparará datos sin borrar señales útiles. Consulta: Jurafsky y Martin, https://web.stanford.edu/~jurafsky/slp3/ ; pandas, https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html . Código y ejemplos originales; datos sintéticos del curso.